In [5]:
from langgraph.graph import StateGraph , START  , END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict  , Annotated
from pydantic import BaseModel , Field
import operator
import warnings
warnings.filterwarnings("ignore")

In [2]:
load_dotenv() 

False

In [ ]:
model = ChatOpenAI(model='gpt-3.5-turbo', temperature=0.7) 

In [3]:
class EvaluationSchema(BaseModel):
    feedback :str = Field(description="Feedback on the essay")
    score : int = Field(description="Score out of 10" , ge=0 , le=10)

In [ ]:
structure_output = model.with_structured_output(EvaluationSchema)

In [8]:
class UPSCState(TypedDict):
    essay : str
    language_feedback : str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback : str
    individual_scores : Annotated[list[int], operator.add]
    avg_score : float

In [14]:
def evaluate_language(state : UPSCState) -> UPSCState:
    prompt = f"Evaluate the following essay for language quality and provide feedback and a score out of 10:\n\n{state['essay']}"
    result = structure_output(prompt)
    language_feedback = result.feedback
    individual_scores = [result.score]
    return {'language_feedback': language_feedback, 'individual_scores': individual_scores}

In [ ]:
def evaluate_analysis(state : UPSCState) -> UPSCState:
    prompt = f"Evaluate the following essay for depth of analysis and provide feedback and a score out of 10:\n\n{state['essay']}"
    result = structure_output(prompt)
    analysis_feedback = result.feedback
    individual_scores.append(result.score)
    return {'analysis_feedback': analysis_feedback, 'individual_scores': individual_scores}

In [11]:
def evaluate_clarity(state : UPSCState) -> UPSCState:
    prompt = f"Evaluate the following essay for clarity and coherence and provide feedback and a score out of 10:\n\n{state['essay']}"
    result = structure_output(prompt)
    clarity_feedback = result.feedback
    individual_scores.append(result.score)
    return {'clarity_feedback': clarity_feedback, 'individual_scores': individual_scores}

In [12]:
def final_evaluation(state : UPSCState) -> UPSCState:
    avg_score = sum(state['individual_scores']) / len(state['individual_scores'])
    overall_feedback = f"Language Feedback: {state['language_feedback']}\nAnalysis Feedback: {state['analysis_feedback']}\nClarity Feedback: {state['clarity_feedback']}\nAverage Score: {avg_score:.2f}"
    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

In [17]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language' ,evaluate_language)
graph.add_node('evaluate_analysis' ,evaluate_analysis)
graph.add_node('evaluate_clarity' ,evaluate_clarity)
graph.add_node('final_evaluation' ,final_evaluation)


graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_clarity')
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_clarity', 'final_evaluation')
graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [16]:
initial_state ={
    'essay': "The impact of climate change on global agriculture is profound and multifaceted. Rising temperatures, changing precipitation patterns, and increased frequency of extreme weather events are affecting crop yields and food security worldwide. Adaptation strategies such as developing drought-resistant crops, improving irrigation systems, and implementing sustainable farming practices are crucial to mitigate these impacts and ensure a stable food supply for the growing global population."
}

In [ ]:
final_State = workflow.invoke(initial_state)
print("Final Feedback:\n", final_State['overall_feedback'])
print("Average Score:", final_State['avg_score'])